In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog, scrolledtext
import pandas as pd
import numpy as np
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.inspection import partial_dependence
import threading
from datetime import datetime
import json
import os
import matplotlib.pyplot as plt
from matplotlib import cm
import warnings
import re
warnings.filterwarnings('ignore')

class CompressiveStrengthPredictor:
    def __init__(self):
        self.setup_style()
        self.setup_paths()
        self.load_data()
        self.train_model()
        self.create_gui()
        
    def setup_style(self):
        # Modern professional color scheme
        self.colors = {
            'primary': '#1B3A4B',        # Deep navy
            'secondary': '#2C5F7A',      # Steel blue
            'accent': '#E8A87C',          # Warm peach
            'background': '#F0F2F5',      # Light gray
            'card_bg': '#FFFFFF',         # White
            'success': '#2E7D32',         # Forest green
            'warning': '#E65100',          # Deep orange
            'danger': '#C62828',           # Deep red
            'info': '#0D47A1',            # Royal blue
            'sandy': '#D4A373',            # Sandy brown
            'soil_brown': '#795548',       # Brown
            'extratrees_blue': '#1565C0',  # Extra Trees Blue
            'extratrees_green': '#2E7D32', # Extra Trees Green
            'gold': '#F9A825',            # Gold
            'purple': '#6A1B9A',          # Purple
        }
        
    def setup_paths(self):
        self.data_path = r"D:\2026 Work\Dr Hariharan Surendran\Paper 1 CS Glass Waste\Revision Work\Data\Data.csv"
        self.save_dir = r"D:\2026 Work\Dr Hariharan Surendran\Paper 1 CS Glass Waste\2nd Revision\PDP"
        os.makedirs(self.save_dir, exist_ok=True)
        self.history_file = os.path.join(self.save_dir, "prediction_history.json")
        
    def clean_column_names(self, columns):
        """Clean column names by removing special characters"""
        cleaned = []
        for col in columns:
            cleaned_col = re.sub(r'[^a-zA-Z0-9_ ]', '_', str(col))
            cleaned_col = re.sub(r'[ _]+', '_', cleaned_col)
            cleaned_col = cleaned_col.strip('_')
            if cleaned_col == '':
                cleaned_col = f'feature_{len(cleaned)}'
            cleaned.append(cleaned_col)
        return cleaned
        
    def load_data(self):
        try:
            print(f"📂 Loading data from: {self.data_path}")
            self.df = pd.read_csv(self.data_path, encoding='ISO-8859-1')
            print(f"✅ Dataset loaded successfully!")
            print(f"📐 Dataset shape: {self.df.shape}")
            
            self.df.columns = self.clean_column_names(self.df.columns)
            print(f"🔧 Cleaned columns: {self.df.columns.tolist()}")
            
            self.predictor_names = [
                'TGWP_replacement_ratio',
                'Water_to_binder_ratio',
                'NaOH_concentration',
                'Curing_age'
            ]
            
            self.found_predictors = {}
            missing_predictors = []
            
            for pred in self.predictor_names:
                found = False
                for col in self.df.columns:
                    col_clean = col.lower().strip()
                    if pred.lower() in col_clean:
                        self.found_predictors[pred] = col
                        found = True
                        print(f"✅ Found '{pred}' → '{col}'")
                        break
                if not found:
                    keywords = {
                        'TGWP_replacement_ratio': ['tgwp', 'replacement', 'glass', 'waste'],
                        'Water_to_binder_ratio': ['water', 'binder', 'w/b', 'ratio'],
                        'NaOH_concentration': ['naoh', 'sodium', 'hydroxide', 'molar'],
                        'Curing_age': ['curing', 'age', 'days']
                    }
                    
                    for col in self.df.columns:
                        col_lower = col.lower()
                        if any(kw in col_lower for kw in keywords.get(pred, [])):
                            self.found_predictors[pred] = col
                            found = True
                            print(f"✅ Found '{pred}' → '{col}' (by keyword)")
                            break
                
                if not found:
                    missing_predictors.append(pred)
                    print(f"⚠️ Could not find '{pred}' in dataset")
            
            if missing_predictors:
                print(f"\n❌ Missing predictors: {missing_predictors}")
                print("Available columns:", self.df.columns.tolist())
                available_cols = [col for col in self.df.columns if col not in ['Compressive_strength_Mpa', 'Compressive strength (Mpa)']]
                for i, pred in enumerate(missing_predictors):
                    if i < len(available_cols):
                        self.found_predictors[pred] = available_cols[i]
                        print(f"⚠️ Using '{available_cols[i]}' as fallback for '{pred}'")
            
            target_col = None
            possible_targets = ['Compressive_strength_Mpa', 'Compressive strength (Mpa)', 
                               'Compressive strength', 'Compressive', 'strength', 
                               'Strength', 'CS', 'cs']
            
            for target in possible_targets:
                for col in self.df.columns:
                    if target.lower() in col.lower():
                        target_col = col
                        break
                if target_col:
                    break

            if target_col is None:
                target_col = self.df.columns[-1]
                print(f"⚠️ Target column not found, using last column: '{target_col}'")
            else:
                print(f"✅ Target column found: '{target_col}'")
            
            self.target_name = target_col
            print(f"🎯 Target variable: '{self.target_name}'")
            
            self.feature_names = list(self.found_predictors.values())
            self.X = self.df[self.feature_names]
            self.y = self.df[target_col]
            
            print(f"\n📊 Using {len(self.feature_names)} predictors:")
            for pred, col in self.found_predictors.items():
                print(f"   • {pred}: '{col}'")
            
            self.feature_stats = {}
            for feature in self.feature_names:
                self.feature_stats[feature] = {
                    'min': self.X[feature].min(),
                    'max': self.X[feature].max(),
                    'mean': self.X[feature].mean(),
                    'std': self.X[feature].std(),
                    'median': self.X[feature].median(),
                    'q1': self.X[feature].quantile(0.25),
                    'q3': self.X[feature].quantile(0.75)
                }
            
            print(f"\n📊 Target Statistics - {self.target_name}:")
            print(f"   • Mean: {self.y.mean():.2f} MPa")
            print(f"   • Std: {self.y.std():.2f} MPa")
            print(f"   • Min: {self.y.min():.2f} MPa")
            print(f"   • Max: {self.y.max():.2f} MPa")
            
        except Exception as e:
            error_msg = f"Failed to load data: {str(e)}"
            print(f"❌ {error_msg}")
            messagebox.showerror("Data Loading Error", error_msg)
            raise
    
    def train_model(self):
        try:
            print("\n" + "="*60)
            print("🌲 TRAINING EXTRA TREES MODEL WITH HYPERPARAMETERS")
            print("="*60)
            
            self.hyperparams = {
                'n_estimators': 100,
                'min_weight_fraction_leaf': 0,
                'min_samples_split': 2,
                'min_samples_leaf': 1,
                'min_impurity_decrease': 0,
                'max_leaf_nodes': None,
                'max_features': 0.7,
                'max_depth': 20,
                'ccp_alpha': 0.001,
                'bootstrap': False,
                'random_state': 42,
                'n_jobs': -1
            }
            
            print("\n📋 Hyperparameters:")
            for param, value in self.hyperparams.items():
                print(f"   • {param:25}: {value}")
            
            self.model = ExtraTreesRegressor(**self.hyperparams)
            
            print("\n🔄 Training in progress...")
            self.model.fit(self.X, self.y)
            print("✅ Extra Trees model training completed!")
            
            self.feature_importance = pd.DataFrame({
                'feature': self.feature_names,
                'importance': self.model.feature_importances_
            }).sort_values('importance', ascending=False)
            
            print(f"\n🏆 FEATURE IMPORTANCE:")
            for i, (_, row) in enumerate(self.feature_importance.iterrows(), 1):
                importance_pct = row['importance'] / self.feature_importance['importance'].sum() * 100
                bar = '█' * int(importance_pct/2) + '░' * (50 - int(importance_pct/2))
                print(f"   {i}. {row['feature'][:30]:30} : {row['importance']:.4f} |{bar}| {importance_pct:.1f}%")
            
            predictions = self.model.predict(self.X)
            
            self.rmse = np.sqrt(np.mean((predictions - self.y) ** 2))
            self.mae = np.mean(np.abs(predictions - self.y))
            self.r2 = 1 - (np.sum((self.y - predictions) ** 2) / np.sum((self.y - self.y.mean()) ** 2))
            
            mask = self.y != 0
            if mask.any():
                mape = np.mean(np.abs((self.y[mask] - predictions[mask]) / self.y[mask])) * 100
            else:
                mape = np.nan
            
            print(f"\n📈 MODEL PERFORMANCE (Internal):")
            print(f"   • R²          : {self.r2:.4f}")
            print(f"   • RMSE        : {self.rmse:.2f} MPa")
            print(f"   • MAE         : {self.mae:.2f} MPa")
            print(f"   • MAPE        : {mape:.2f}%")
            print(f"   • Model       : Extra Trees Regressor (Tuned)")
            
        except Exception as e:
            error_msg = f"Failed to train model: {str(e)}"
            print(f"❌ {error_msg}")
            messagebox.showerror("Model Training Error", error_msg)
            raise
    
    def create_gui(self):
        self.root = tk.Tk()
        self.root.title("🏗️ Compressive Strength Predictor - Extra Trees (Tuned)")
        self.root.geometry("1600x1000")
        self.root.configure(bg=self.colors['background'])
        self.root.minsize(1200, 800)
        
        # Center window
        self.root.update_idletasks()
        width = self.root.winfo_width()
        height = self.root.winfo_height()
        x = (self.root.winfo_screenwidth() // 2) - (width // 2)
        y = (self.root.winfo_screenheight() // 2) - (height // 2)
        self.root.geometry(f'1600x1000+{x}+{y}')
        
        # Configure ttk styles
        self.setup_styles()
        
        # Create main container with padding
        main_container = ttk.Frame(self.root)
        main_container.pack(fill='both', expand=True, padx=10, pady=10)
        
        # Create header
        self.create_header(main_container)
        
        # Create notebook for tabs
        self.notebook = ttk.Notebook(main_container)
        self.notebook.pack(fill='both', expand=True, pady=(10, 0))
        
        # Create tabs
        self.prediction_tab = self.create_prediction_tab()
        self.analysis_tab = self.create_analysis_tab()
        self.history_tab = self.create_history_tab()
        self.model_info_tab = self.create_model_info_tab()
        
        # Add tabs to notebook
        self.notebook.add(self.prediction_tab, text="🔮 Prediction")
        self.notebook.add(self.analysis_tab, text="📊 Analysis")
        self.notebook.add(self.history_tab, text="📋 History")
        self.notebook.add(self.model_info_tab, text="ℹ️ Model Info")
        
        # Initialize prediction history
        self.prediction_history = []
        self.load_history()
        
        # Create status bar
        self.create_status_bar(main_container)
        
        print("\n✅ GUI created successfully!")
        print("🏁 Application ready for predictions\n")
    
    def create_header(self, parent):
        """Create application header with branding"""
        header_frame = tk.Frame(parent, bg=self.colors['primary'], height=80)
        header_frame.pack(fill='x', pady=(0, 10))
        header_frame.pack_propagate(False)
        
        # Left section - Logo and title
        left_frame = tk.Frame(header_frame, bg=self.colors['primary'])
        left_frame.pack(side='left', padx=20, pady=10, fill='y')
        
        title_label = tk.Label(left_frame, 
                              text="🏗️ Concrete Strength Predictor",
                              font=("Segoe UI", 20, "bold"),
                              fg='white', bg=self.colors['primary'])
        title_label.pack(side='left')
        
        subtitle_label = tk.Label(left_frame,
                                 text="Extra Trees Regressor (Tuned)",
                                 font=("Segoe UI", 11),
                                 fg=self.colors['accent'], bg=self.colors['primary'])
        subtitle_label.pack(side='left', padx=(10, 0))
        
        # Right section - Model info without R²
        right_frame = tk.Frame(header_frame, bg=self.colors['primary'])
        right_frame.pack(side='right', padx=20, pady=10)
        
        # Model badges - removed R²
        badges = [
            (f"Predictors: {len(self.feature_names)}", self.colors['info']),
            (f"Samples: {len(self.y)}", self.colors['success']),
        ]
        
        for text, color in badges:
            badge = tk.Label(right_frame, text=text,
                           bg=color, fg='white',
                           font=("Segoe UI", 10, "bold"),
                           padx=12, pady=5,
                           relief='raised', bd=1)
            badge.pack(side='left', padx=5)
    
    def create_status_bar(self, parent):
        """Create status bar at bottom"""
        status_frame = tk.Frame(parent, bg=self.colors['secondary'], height=30)
        status_frame.pack(fill='x', pady=(10, 0))
        status_frame.pack_propagate(False)
        
        # Left status
        left_status = tk.Label(status_frame,
                              text="✅ Ready for predictions",
                              font=("Segoe UI", 9),
                              fg='white', bg=self.colors['secondary'])
        left_status.pack(side='left', padx=15)
        
        # Right status - timestamp
        right_status = tk.Label(status_frame,
                               text=f"Model loaded: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
                               font=("Segoe UI", 9),
                               fg='white', bg=self.colors['secondary'])
        right_status.pack(side='right', padx=15)
        
        # Store references for updates
        self.status_left = left_status
        self.status_right = right_status
    
    def setup_styles(self):
        style = ttk.Style()
        style.theme_use('clam')
        
        # Configure custom styles
        style.configure('Accent.TButton',
                       background=self.colors['accent'],
                       foreground='white',
                       font=('Segoe UI', 12, 'bold'),
                       borderwidth=2,
                       relief='raised',
                       padding=10)
        
        style.map('Accent.TButton',
                 background=[('active', '#D4956B'), ('pressed', '#B8835A')])
        
        style.configure('Success.TButton',
                       background=self.colors['success'],
                       foreground='white',
                       font=('Segoe UI', 11, 'bold'),
                       borderwidth=2,
                       padding=8)
        
        style.map('Success.TButton',
                 background=[('active', '#1B5E20'), ('pressed', '#0D3B12')])
        
        style.configure('Primary.TButton',
                       background=self.colors['primary'],
                       foreground='white',
                       font=('Segoe UI', 10, 'bold'),
                       padding=6)
        
        style.map('Primary.TButton',
                 background=[('active', '#1E4A5F'), ('pressed', '#0D2B3A')])
        
        style.configure('Card.TFrame',
                       background=self.colors['card_bg'],
                       relief='raised',
                       borderwidth=2)
        
        style.configure('Header.TLabel',
                       font=('Segoe UI', 14, 'bold'),
                       foreground=self.colors['primary'])
        
        style.configure('SubHeader.TLabel',
                       font=('Segoe UI', 11, 'bold'),
                       foreground=self.colors['soil_brown'])
    
    def create_prediction_tab(self):
        tab = ttk.Frame(self.notebook)
        
        # Main container with two panels
        main_paned = ttk.PanedWindow(tab, orient=tk.HORIZONTAL)
        main_paned.pack(fill='both', expand=True, padx=10, pady=10)
        
        # Left panel - Inputs (40% width)
        left_panel = ttk.Frame(main_paned, style='Card.TFrame')
        main_paned.add(left_panel, weight=40)
        
        # Right panel - Results (60% width)
        right_panel = ttk.Frame(main_paned, style='Card.TFrame')
        main_paned.add(right_panel, weight=60)
        
        # ========== LEFT PANEL: Input Parameters ==========
        self.create_input_panel(left_panel)
        
        # ========== RIGHT PANEL: Results ==========
        self.create_results_panel(right_panel)
        
        return tab
    
    def create_input_panel(self, parent):
        """Create the input parameters panel"""
        # Header
        header_frame = tk.Frame(parent, bg=self.colors['card_bg'])
        header_frame.pack(fill='x', pady=(10, 5), padx=15)
        
        tk.Label(header_frame, text="📊 Mix Design Parameters",
                font=("Segoe UI", 16, "bold"),
                fg=self.colors['primary'], bg=self.colors['card_bg']).pack(anchor='w')
        
        tk.Label(header_frame, text="Enter four validated mix parameters for compressive strength prediction",
                font=("Segoe UI", 10),
                fg=self.colors['secondary'], bg=self.colors['card_bg']).pack(anchor='w')
        
        # Quick selection section
        quick_frame = tk.LabelFrame(parent, text="⚡ Quick Actions", 
                                   font=("Segoe UI", 10, "bold"),
                                   fg=self.colors['primary'],
                                   bg=self.colors['card_bg'],
                                   padx=15, pady=10)
        quick_frame.pack(fill='x', padx=15, pady=10)
        
        # Preset selection
        preset_row = tk.Frame(quick_frame, bg=self.colors['card_bg'])
        preset_row.pack(fill='x', pady=(0, 10))
        
        tk.Label(preset_row, text="Mix Design Preset:",
                font=("Segoe UI", 10, "bold"),
                bg=self.colors['card_bg']).pack(side='left', padx=(0, 10))
        
        self.preset_var = tk.StringVar(value="Medium Strength Mix")
        preset_combo = ttk.Combobox(preset_row, textvariable=self.preset_var,
                                   values=["Low Strength Mix", "Medium Strength Mix", 
                                           "High Strength Mix", "Very High Strength Mix",
                                           "Custom"],
                                   state="readonly", width=20)
        preset_combo.pack(side='left')
        preset_combo.bind("<<ComboboxSelected>>", self.load_preset)
        
        # Quick action buttons
        action_frame = tk.Frame(quick_frame, bg=self.colors['card_bg'])
        action_frame.pack(fill='x')
        
        actions = [
            ("🎲 Random", self.fill_random_sample),
            ("📊 Min", self.fill_min_strength),
            ("📈 Max", self.fill_max_strength),
            ("🔄 Clear", self.clear_inputs)
        ]
        
        for text, cmd in actions:
            btn = tk.Button(action_frame, text=text, command=cmd,
                          bg=self.colors['secondary'], fg='white',
                          font=("Segoe UI", 9, "bold"),
                          padx=10, pady=5, relief='raised', bd=1,
                          cursor='hand2')
            btn.pack(side='left', padx=3, expand=True, fill='x')
        
        # Input fields section
        input_frame = tk.LabelFrame(parent, text="📝 Parameter Values",
                                   font=("Segoe UI", 10, "bold"),
                                   fg=self.colors['primary'],
                                   bg=self.colors['card_bg'],
                                   padx=15, pady=10)
        input_frame.pack(fill='both', expand=True, padx=15, pady=5)
        
        # Create canvas with scrollbar for inputs
        canvas = tk.Canvas(input_frame, highlightthickness=0, bg=self.colors['card_bg'])
        scrollbar = ttk.Scrollbar(input_frame, orient="vertical", command=canvas.yview)
        scrollable_frame = tk.Frame(canvas, bg=self.colors['card_bg'])
        
        scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )
        
        canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)
        
        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")
        
        # Bind mousewheel for scrolling
        def _on_mousewheel(event):
            canvas.yview_scroll(int(-1*(event.delta/120)), "units")
        
        canvas.bind_all("<MouseWheel>", _on_mousewheel)
        
        # Feature entries with descriptive labels
        self.entries = {}
        
        display_info = {
            'TGWP_replacement_ratio': {
                'label': 'TGWP Replacement Ratio',
                'unit': '%',
                'desc': 'Glass waste powder replacement',
                'icon': '🔄'
            },
            'Water_to_binder_ratio': {
                'label': 'Water-to-Binder Ratio',
                'unit': '',
                'desc': 'Water to binder ratio',
                'icon': '💧'
            },
            'NaOH_concentration': {
                'label': 'NaOH Concentration',
                'unit': 'M',
                'desc': 'Sodium hydroxide molarity',
                'icon': '🧪'
            },
            'Curing_age': {
                'label': 'Curing Age',
                'unit': 'days',
                'desc': 'Concrete curing duration',
                'icon': '⏳'
            }
        }
        
        for feature in self.feature_names:
            feat_frame = tk.Frame(scrollable_frame, bg=self.colors['card_bg'])
            feat_frame.pack(fill='x', pady=8)
            
            stats = self.feature_stats[feature]
            importance = self.feature_importance[self.feature_importance['feature'] == feature]['importance'].values[0]
            importance_pct = importance / self.feature_importance['importance'].sum() * 100
            
            # Find which predictor this feature corresponds to
            pred_key = None
            for key, col in self.found_predictors.items():
                if col == feature:
                    pred_key = key
                    break
            
            info = display_info.get(pred_key, {'label': feature, 'unit': '', 'desc': '', 'icon': '📊'})
            
            # Feature name with importance indicator
            name_frame = tk.Frame(feat_frame, bg=self.colors['card_bg'])
            name_frame.pack(fill='x')
            
            # Feature label with icon
            label_text = f"{info['icon']} {info['label']}"
            if info['desc']:
                label_text += f"\n   {info['desc']}"
            
            label = tk.Label(name_frame,
                            text=label_text,
                            font=("Segoe UI", 9, "bold"),
                            fg=self.colors['primary'],
                            bg=self.colors['card_bg'],
                            justify='left')
            label.pack(side='left')
            
            # Importance bar
            bar_frame = tk.Frame(name_frame, bg=self.colors['card_bg'])
            bar_frame.pack(side='right', fill='x', expand=True, padx=(10, 0))
            
            # Create custom progress bar for importance
            bar_canvas = tk.Canvas(bar_frame, height=12, bg=self.colors['card_bg'], highlightthickness=0)
            bar_canvas.pack(fill='x', pady=2)
            
            # Draw importance bar
            bar_width = 150
            fill_width = int(bar_width * importance_pct / 100)
            
            # Background
            bar_canvas.create_rectangle(0, 0, bar_width, 12, fill=self.colors['background'], outline='')
            # Filled portion
            color = self.get_importance_color(importance)
            bar_canvas.create_rectangle(0, 0, fill_width, 12, fill=color, outline='')
            # Percentage text
            bar_canvas.create_text(bar_width // 2, 6, 
                                  text=f"{importance_pct:.1f}%",
                                  font=("Segoe UI", 7, "bold"),
                                  fill='white')
            
            # Input row
            input_row = tk.Frame(feat_frame, bg=self.colors['card_bg'])
            input_row.pack(fill='x', pady=(5, 0))
            
            # Entry field
            entry = ttk.Entry(input_row, width=12, font=("Segoe UI", 10))
            entry.insert(0, f"{stats['mean']:.3f}")
            entry.pack(side='left', padx=(0, 8))
            
            # Range label
            range_label = tk.Label(input_row,
                                  text=f"[{stats['min']:.2f} - {stats['max']:.2f}]",
                                  font=("Segoe UI", 8),
                                  fg=self.colors['secondary'],
                                  bg=self.colors['card_bg'])
            range_label.pack(side='left', padx=5)
            
            # Units
            if info['unit']:
                unit_label = tk.Label(input_row,
                                     text=info['unit'],
                                     font=("Segoe UI", 9, "bold"),
                                     fg=self.colors['soil_brown'],
                                     bg=self.colors['card_bg'])
                unit_label.pack(side='left')
            
            self.entries[feature] = entry
            
            # Slider
            if stats['max'] - stats['min'] > 0:
                slider = ttk.Scale(input_row,
                                  from_=stats['min'],
                                  to=stats['max'],
                                  value=stats['mean'],
                                  orient='horizontal',
                                  length=120,
                                  command=lambda v, f=feature: self.update_entry_from_slider(f, float(v)))
                slider.pack(side='right', padx=5)
        
        # Predict button
        predict_frame = tk.Frame(parent, bg=self.colors['card_bg'])
        predict_frame.pack(fill='x', padx=15, pady=15)
        
        self.predict_btn = tk.Button(predict_frame,
                                    text="🔮 PREDICT COMPRESSIVE STRENGTH",
                                    command=self.threaded_predict,
                                    bg=self.colors['accent'],
                                    fg='white',
                                    font=("Segoe UI", 14, "bold"),
                                    padx=20, pady=12,
                                    relief='raised', bd=3,
                                    cursor='hand2')
        self.predict_btn.pack(fill='x')
    
    def create_results_panel(self, parent):
        """Create the results display panel"""
        # Header
        header_frame = tk.Frame(parent, bg=self.colors['card_bg'])
        header_frame.pack(fill='x', pady=(10, 5), padx=15)
        
        tk.Label(header_frame, text="📈 Prediction Results",
                font=("Segoe UI", 16, "bold"),
                fg=self.colors['primary'], bg=self.colors['card_bg']).pack(anchor='w')
        
        tk.Label(header_frame, text="Compressive strength prediction results and analysis",
                font=("Segoe UI", 10),
                fg=self.colors['secondary'], bg=self.colors['card_bg']).pack(anchor='w')
        
        # Main result display - Large card
        result_card = tk.Frame(parent, bg=self.colors['card_bg'],
                              relief='raised', bd=2)
        result_card.pack(fill='x', padx=15, pady=10)
        
        # Strength value
        strength_frame = tk.Frame(result_card, bg=self.colors['card_bg'])
        strength_frame.pack(pady=20)
        
        tk.Label(strength_frame, text="Compressive Strength",
                font=("Segoe UI", 14), fg=self.colors['secondary'],
                bg=self.colors['card_bg']).pack()
        
        self.result_var = tk.StringVar(value="---")
        result_label = tk.Label(strength_frame,
                               textvariable=self.result_var,
                               font=("Segoe UI", 56, "bold"),
                               fg=self.colors['primary'],
                               bg=self.colors['card_bg'])
        result_label.pack()
        
        tk.Label(strength_frame, text="MPa",
                font=("Segoe UI", 14),
                fg=self.colors['secondary'],
                bg=self.colors['card_bg']).pack()
        
        # Confidence and classification row
        info_row = tk.Frame(result_card, bg=self.colors['card_bg'])
        info_row.pack(fill='x', padx=20, pady=(0, 20))
        
        # Confidence meter
        conf_frame = tk.Frame(info_row, bg=self.colors['card_bg'])
        conf_frame.pack(side='left', expand=True, fill='x')
        
        tk.Label(conf_frame, text="Confidence Level",
                font=("Segoe UI", 10), fg=self.colors['secondary'],
                bg=self.colors['card_bg']).pack(anchor='w')
        
        self.confidence_var = tk.StringVar(value="--%")
        conf_value = tk.Label(conf_frame,
                             textvariable=self.confidence_var,
                             font=("Segoe UI", 14, "bold"),
                             fg=self.colors['success'],
                             bg=self.colors['card_bg'])
        conf_value.pack(anchor='w')
        
        # Confidence bar
        self.confidence_bar = ttk.Progressbar(conf_frame,
                                             length=200,
                                             mode='determinate')
        self.confidence_bar.pack(anchor='w', pady=5)
        self.confidence_bar['value'] = 0
        
        # Classification
        class_frame = tk.Frame(info_row, bg=self.colors['card_bg'])
        class_frame.pack(side='right', expand=True, fill='x')
        
        tk.Label(class_frame, text="Strength Classification",
                font=("Segoe UI", 10), fg=self.colors['secondary'],
                bg=self.colors['card_bg']).pack(anchor='w')
        
        self.classification_var = tk.StringVar(value="Not predicted")
        self.classification_label = tk.Label(class_frame,
                                            textvariable=self.classification_var,
                                            font=("Segoe UI", 16, "bold"),
                                            bg=self.colors['card_bg'])
        self.classification_label.pack(anchor='w')
        
        # Detailed results notebook
        detail_notebook = ttk.Notebook(parent)
        detail_notebook.pack(fill='both', expand=True, padx=15, pady=10)
        
        # Tab 1: Feature Contributions
        contributions_tab = tk.Frame(detail_notebook, bg=self.colors['card_bg'])
        detail_notebook.add(contributions_tab, text="📊 Feature Contributions")
        
        self.contributions_text = scrolledtext.ScrolledText(contributions_tab,
                                                           font=("Segoe UI", 10),
                                                           wrap=tk.WORD,
                                                           height=10,
                                                           bg='white',
                                                           relief='solid',
                                                           bd=1)
        self.contributions_text.pack(fill='both', expand=True, padx=5, pady=5)
        self.contributions_text.insert(1.0, "Feature contributions will appear here after prediction.\n\n"
                                            "Top features and their impact on compressive strength:")
        self.contributions_text.config(state='disabled')
        
        # Tab 2: Comparison
        comparison_tab = tk.Frame(detail_notebook, bg=self.colors['card_bg'])
        detail_notebook.add(comparison_tab, text="📈 Comparison")
        
        self.comparison_text = scrolledtext.ScrolledText(comparison_tab,
                                                        font=("Segoe UI", 10),
                                                        wrap=tk.WORD,
                                                        height=10,
                                                        bg='white',
                                                        relief='solid',
                                                        bd=1)
        self.comparison_text.pack(fill='both', expand=True, padx=5, pady=5)
        self.comparison_text.insert(1.0, "Comparison with typical values will appear here.")
        self.comparison_text.config(state='disabled')
        
        # Action buttons
        action_frame = tk.Frame(parent, bg=self.colors['card_bg'])
        action_frame.pack(fill='x', padx=15, pady=10)
        
        action_buttons = [
            ("💾 Save Result", self.save_result),
            ("📋 Copy", self.copy_to_clipboard),
            ("📤 Export", self.export_results),
            ("📊 Analyze", lambda: self.notebook.select(1))
        ]
        
        for text, cmd in action_buttons:
            btn = tk.Button(action_frame, text=text, command=cmd,
                          bg=self.colors['secondary'], fg='white',
                          font=("Segoe UI", 10, "bold"),
                          padx=10, pady=6,
                          relief='raised', bd=1,
                          cursor='hand2')
            btn.pack(side='left', padx=3, expand=True, fill='x')
    
    def create_analysis_tab(self):
        tab = ttk.Frame(self.notebook)
        
        # Analysis controls
        control_frame = tk.LabelFrame(tab, text="📊 Analysis Tools",
                                     font=("Segoe UI", 12, "bold"),
                                     fg=self.colors['primary'],
                                     bg=self.colors['card_bg'],
                                     padx=20, pady=15)
        control_frame.pack(fill='x', padx=20, pady=10)
        
        # Analysis buttons grid
        button_grid = tk.Frame(control_frame, bg=self.colors['card_bg'])
        button_grid.pack()
        
        analysis_buttons = [
            ("🏆 Feature Importance", self.plot_feature_importance, self.colors['primary']),
            ("📊 Data Distribution", self.plot_data_distribution, self.colors['success']),
            ("📈 Partial Dependence", self.plot_partial_dependence, self.colors['warning']),
            ("🔗 Correlation Matrix", self.plot_correlation_matrix, self.colors['danger']),
            ("📋 Feature Statistics", self.plot_feature_statistics, self.colors['info']),
            ("🎯 Actual vs Predicted", self.plot_actual_vs_predicted, self.colors['extratrees_green']),
            ("🏗️ Mix Parameters", self.plot_mix_parameters, self.colors['sandy']),
            ("📉 Residual Analysis", self.plot_residual_analysis, self.colors['purple']),
        ]
        
        for i, (text, command, color) in enumerate(analysis_buttons):
            row, col = divmod(i, 4)
            btn = tk.Button(button_grid, text=text, command=command,
                          bg=color, fg='white',
                          font=("Segoe UI", 10, "bold"),
                          padx=12, pady=8,
                          relief='raised', bd=2,
                          cursor='hand2', width=20)
            btn.grid(row=row, column=col, padx=5, pady=5)
            
            # Hover effect
            btn.bind("<Enter>", lambda e, b=btn, c=color: b.config(bg=self.lighten_color(c)))
            btn.bind("<Leave>", lambda e, b=btn, c=color: b.config(bg=c))
        
        # Save plots button
        save_frame = tk.Frame(control_frame, bg=self.colors['card_bg'])
        save_frame.pack(pady=(15, 0))
        
        tk.Button(save_frame, text="💾 SAVE ALL PLOTS",
                 command=self.save_all_plots,
                 bg=self.colors['soil_brown'], fg='white',
                 font=("Segoe UI", 12, "bold"),
                 padx=30, pady=10,
                 relief='raised', bd=3,
                 cursor='hand2').pack()
        
        # Plot area
        self.analysis_frame = tk.Frame(tab, bg=self.colors['background'])
        self.analysis_frame.pack(fill='both', expand=True, padx=20, pady=10)
        
        # Initial plot
        self.plot_feature_importance()
        
        return tab
    
    def create_history_tab(self):
        tab = ttk.Frame(self.notebook)
        
        # Header
        header_frame = tk.Frame(tab, bg=self.colors['card_bg'])
        header_frame.pack(fill='x', padx=20, pady=10)
        
        tk.Label(header_frame, text="📋 Prediction History",
                font=("Segoe UI", 18, "bold"),
                fg=self.colors['primary'],
                bg=self.colors['card_bg']).pack(anchor='w')
        
        tk.Label(header_frame, text="Record of all predictions made with this model",
                font=("Segoe UI", 10),
                fg=self.colors['secondary'],
                bg=self.colors['card_bg']).pack(anchor='w')
        
        # History controls
        control_frame = tk.LabelFrame(tab, text="History Management",
                                     font=("Segoe UI", 10, "bold"),
                                     fg=self.colors['primary'],
                                     bg=self.colors['card_bg'],
                                     padx=15, pady=10)
        control_frame.pack(fill='x', padx=20, pady=10)
        
        button_frame = tk.Frame(control_frame, bg=self.colors['card_bg'])
        button_frame.pack()
        
        history_buttons = [
            ("🗑️ Clear History", self.clear_history, self.colors['danger']),
            ("💾 Export to CSV", self.export_history, self.colors['success']),
            ("📤 Load History", self.load_history_from_file, self.colors['info']),
            ("🔄 Refresh", self.update_history_display, self.colors['primary']),
        ]
        
        for text, cmd, color in history_buttons:
            btn = tk.Button(button_frame, text=text, command=cmd,
                          bg=color, fg='white',
                          font=("Segoe UI", 10, "bold"),
                          padx=12, pady=6,
                          relief='raised', bd=2,
                          cursor='hand2')
            btn.pack(side='left', padx=5)
        
        # Statistics
        self.history_stats_var = tk.StringVar(value="No predictions yet")
        stats_label = tk.Label(control_frame,
                              textvariable=self.history_stats_var,
                              font=("Segoe UI", 11, "bold"),
                              fg=self.colors['success'],
                              bg=self.colors['card_bg'])
        stats_label.pack(pady=(10, 0))
        
        # History display
        display_frame = tk.Frame(tab, bg=self.colors['background'])
        display_frame.pack(fill='both', expand=True, padx=20, pady=(0, 20))
        
        # Create treeview for history
        columns = ('Timestamp', 'Strength (MPa)', 'Classification', 'Confidence', 'Top Feature')
        self.history_tree = ttk.Treeview(display_frame, columns=columns, show='headings', height=20)
        
        # Define headings
        self.history_tree.heading('Timestamp', text='Date & Time')
        self.history_tree.heading('Strength (MPa)', text='Strength (MPa)')
        self.history_tree.heading('Classification', text='Classification')
        self.history_tree.heading('Confidence', text='Confidence')
        self.history_tree.heading('Top Feature', text='Top Feature')
        
        # Define column widths
        self.history_tree.column('Timestamp', width=150)
        self.history_tree.column('Strength (MPa)', width=120)
        self.history_tree.column('Classification', width=150)
        self.history_tree.column('Confidence', width=100)
        self.history_tree.column('Top Feature', width=150)
        
        # Add scrollbars
        vsb = ttk.Scrollbar(display_frame, orient="vertical", command=self.history_tree.yview)
        hsb = ttk.Scrollbar(display_frame, orient="horizontal", command=self.history_tree.xview)
        self.history_tree.configure(yscrollcommand=vsb.set, xscrollcommand=hsb.set)
        
        # Grid layout
        self.history_tree.grid(row=0, column=0, sticky='nsew')
        vsb.grid(row=0, column=1, sticky='ns')
        hsb.grid(row=1, column=0, sticky='ew')
        
        display_frame.grid_rowconfigure(0, weight=1)
        display_frame.grid_columnconfigure(0, weight=1)
        
        # Bind double-click
        self.history_tree.bind('<Double-Button-1>', self.load_history_entry)
        
        return tab
    
    def create_model_info_tab(self):
        tab = ttk.Frame(self.notebook)
        
        # Main container with scrollbar
        canvas = tk.Canvas(tab, bg=self.colors['background'], highlightthickness=0)
        scrollbar = ttk.Scrollbar(tab, orient="vertical", command=canvas.yview)
        scrollable_frame = tk.Frame(canvas, bg=self.colors['background'])
        
        scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
        )
        
        canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)
        
        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")
        
        # Model Information
        info_frame = tk.LabelFrame(scrollable_frame, text="🌲 Model Information",
                                  font=("Segoe UI", 12, "bold"),
                                  fg=self.colors['primary'],
                                  bg=self.colors['card_bg'],
                                  padx=20, pady=15)
        info_frame.pack(fill='x', padx=20, pady=10)
        
        # Model overview - removed R² and other metrics from display
        overview_text = f"""
═══════════════════════════════════════════════════════════════
              COMPRESSIVE STRENGTH PREDICTOR
═══════════════════════════════════════════════════════════════

🎯 Target Variable: {self.target_name}
📊 Number of Predictors: {len(self.feature_names)}
📈 Training Samples: {len(self.X)}
🌲 Algorithm: Extra Trees Regressor (Tuned)

📋 Predictors Used:
"""
        
        for pred, col in self.found_predictors.items():
            overview_text += f"   • {pred}: '{col}'\n"
        
        overview_text += """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                    HYPERPARAMETERS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""
        
        for param, value in self.hyperparams.items():
            overview_text += f"• {param:25}: {value}\n"
        
        overview_text += """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""
        
        model_info_label = tk.Label(info_frame, text=overview_text,
                                   font=("Courier", 10),
                                   justify='left',
                                   bg='white',
                                   padx=20, pady=20,
                                   relief='solid', bd=1)
        model_info_label.pack(fill='x')
        
        # Feature Importance Table
        importance_frame = tk.LabelFrame(scrollable_frame, text="📊 Feature Importance Ranking",
                                        font=("Segoe UI", 12, "bold"),
                                        fg=self.colors['primary'],
                                        bg=self.colors['card_bg'],
                                        padx=20, pady=15)
        importance_frame.pack(fill='x', padx=20, pady=10)
        
        # Create treeview for feature importance
        columns = ('Rank', 'Feature', 'Importance', 'Percentage', 'Cumulative')
        importance_tree = ttk.Treeview(importance_frame, columns=columns, show='headings', height=8)
        
        importance_tree.heading('Rank', text='Rank')
        importance_tree.heading('Feature', text='Feature Name')
        importance_tree.heading('Importance', text='Importance Score')
        importance_tree.heading('Percentage', text='Percentage (%)')
        importance_tree.heading('Cumulative', text='Cumulative %')
        
        importance_tree.column('Rank', width=50)
        importance_tree.column('Feature', width=250)
        importance_tree.column('Importance', width=120)
        importance_tree.column('Percentage', width=100)
        importance_tree.column('Cumulative', width=100)
        
        # Add scrollbar
        tree_scroll = ttk.Scrollbar(importance_frame, orient="vertical", command=importance_tree.yview)
        importance_tree.configure(yscrollcommand=tree_scroll.set)
        
        importance_tree.pack(side='left', fill='both', expand=True)
        tree_scroll.pack(side='right', fill='y')
        
        # Add data
        total_importance = self.feature_importance['importance'].sum()
        cumulative = 0
        
        for i, (_, row) in enumerate(self.feature_importance.iterrows(), 1):
            percentage = (row['importance'] / total_importance) * 100
            cumulative += percentage
            # Get display name
            display_name = row['feature']
            for pred, col in self.found_predictors.items():
                if col == row['feature']:
                    display_info = {
                        'TGWP_replacement_ratio': 'TGWP Replacement',
                        'Water_to_binder_ratio': 'Water-to-Binder Ratio',
                        'NaOH_concentration': 'NaOH Concentration',
                        'Curing_age': 'Curing Age'
                    }
                    display_name = display_info.get(pred, row['feature'])
                    break
            
            importance_tree.insert('', 'end', values=(
                i,
                display_name,
                f"{row['importance']:.6f}",
                f"{percentage:.2f}%",
                f"{cumulative:.2f}%"
            ))
        
        # Strength Classification Guide
        guide_frame = tk.LabelFrame(scrollable_frame, text="📋 Strength Classification Guide",
                                   font=("Segoe UI", 12, "bold"),
                                   fg=self.colors['primary'],
                                   bg=self.colors['card_bg'],
                                   padx=20, pady=15)
        guide_frame.pack(fill='x', padx=20, pady=10)
        
        guide_text = """
═══════════════════════════════════════════════════════════════
              COMPRESSIVE STRENGTH CLASSIFICATION
═══════════════════════════════════════════════════════════════

Based on the predicted compressive strength (MPa):

🟢 VERY LOW STRENGTH  : < 20 MPa   - Lightweight, non-structural applications
🟡 LOW STRENGTH       : 20-30 MPa   - General construction, foundations
🟠 MEDIUM STRENGTH    : 30-40 MPa   - Structural concrete, beams, columns
🔴 HIGH STRENGTH      : 40-60 MPa   - High-rise buildings, bridges
🔥 VERY HIGH STRENGTH : > 60 MPa    - Special structures, prestressed concrete

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                VALIDATED PREDICTOR RANGES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

• TGWP Replacement Ratio : 0-30% (Optimal range: 10-20%)
• Water-to-Binder Ratio  : 0.35-0.55 (Lower ratio → Higher strength)
• NaOH Concentration     : 0-10 M (Optimal: 6-8 M)
• Curing Age            : 7-28+ days (Strength increases with age)
"""
        
        guide_label = tk.Label(guide_frame, text=guide_text,
                             font=("Courier", 10),
                             justify='left',
                             bg='white',
                             padx=20, pady=20,
                             relief='solid', bd=1)
        guide_label.pack(fill='x')
        
        return tab
    
    def get_importance_color(self, importance):
        if importance > 0.1:
            return self.colors['danger']
        elif importance > 0.05:
            return self.colors['warning']
        elif importance > 0.02:
            return self.colors['success']
        else:
            return self.colors['primary']
    
    def lighten_color(self, color):
        """Lighten a color for hover effect"""
        lightened = {
            self.colors['primary']: '#2A5A6B',
            self.colors['secondary']: '#4A7A9A',
            self.colors['success']: '#4A9A5A',
            self.colors['warning']: '#E87A20',
            self.colors['danger']: '#D84A3A',
            self.colors['sandy']: '#E6B88A',
            self.colors['info']: '#2A6AAA',
            self.colors['purple']: '#8A3AAA',
            self.colors['soil_brown']: '#A67B5B',
            self.colors['extratrees_blue']: '#3C9BFF',
            self.colors['extratrees_green']: '#5CB85C',
        }
        return lightened.get(color, color)
    
    def load_preset(self, event=None):
        preset = self.preset_var.get()
        self.load_preset_by_name(preset)
    
    def load_preset_by_name(self, preset_name):
        # Define presets based on concrete strength characteristics
        presets = {
            "Low Strength Mix": {
                "description": "Low strength concrete (< 20 MPa) for non-structural applications",
                "values": {
                    'TGWP_replacement_ratio': 20,
                    'Water_to_binder_ratio': 0.55,
                    'NaOH_concentration': 4,
                    'Curing_age': 7
                }
            },
            "Medium Strength Mix": {
                "description": "Medium strength concrete (20-30 MPa) for general construction",
                "values": {
                    'TGWP_replacement_ratio': 10,
                    'Water_to_binder_ratio': 0.45,
                    'NaOH_concentration': 6,
                    'Curing_age': 14
                }
            },
            "High Strength Mix": {
                "description": "High strength concrete (30-40 MPa) for structural elements",
                "values": {
                    'TGWP_replacement_ratio': 5,
                    'Water_to_binder_ratio': 0.38,
                    'NaOH_concentration': 8,
                    'Curing_age': 28
                }
            },
            "Very High Strength Mix": {
                "description": "Very high strength concrete (> 40 MPa) for special structures",
                "values": {
                    'TGWP_replacement_ratio': 15,
                    'Water_to_binder_ratio': 0.35,
                    'NaOH_concentration': 10,
                    'Curing_age': 56
                }
            }
        }
        
        if preset_name in presets:
            preset_values = presets[preset_name]["values"]
            
            for feature in self.entries:
                for pred, col in self.found_predictors.items():
                    if col == feature:
                        if pred in preset_values:
                            value = preset_values[pred]
                            stats = self.feature_stats[feature]
                            value = max(stats['min'], min(stats['max'], value))
                            self.entries[feature].delete(0, tk.END)
                            self.entries[feature].insert(0, f"{value:.3f}")
                            
                            # Update slider
                            for widget in self.entries[feature].master.winfo_children():
                                if isinstance(widget, ttk.Scale):
                                    widget.set(value)
                        break
            
            self.status_left.config(text=f"✅ Loaded preset: {preset_name}")
    
    def fill_random_sample(self):
        random_idx = np.random.randint(0, len(self.df))
        sample = self.df.iloc[random_idx]
        
        for feature in self.entries:
            if feature in sample:
                self.entries[feature].delete(0, tk.END)
                value = sample[feature]
                self.entries[feature].insert(0, f"{value:.3f}")
                # Update slider
                for widget in self.entries[feature].master.winfo_children():
                    if isinstance(widget, ttk.Scale):
                        widget.set(value)
        
        self.status_left.config(text=f"✅ Loaded random sample #{random_idx}")
    
    def fill_min_strength(self):
        best_idx = self.y.idxmin()
        sample = self.df.iloc[best_idx]
        
        for feature in self.entries:
            if feature in sample:
                self.entries[feature].delete(0, tk.END)
                self.entries[feature].insert(0, f"{sample[feature]:.3f}")
                # Update slider
                for widget in self.entries[feature].master.winfo_children():
                    if isinstance(widget, ttk.Scale):
                        widget.set(sample[feature])
        
        self.status_left.config(text=f"📊 Loaded minimum strength scenario ({self.y.min():.2f} MPa)")
    
    def fill_max_strength(self):
        worst_idx = self.y.idxmax()
        sample = self.df.iloc[worst_idx]
        
        for feature in self.entries:
            if feature in sample:
                self.entries[feature].delete(0, tk.END)
                self.entries[feature].insert(0, f"{sample[feature]:.3f}")
                # Update slider
                for widget in self.entries[feature].master.winfo_children():
                    if isinstance(widget, ttk.Scale):
                        widget.set(sample[feature])
        
        self.status_left.config(text=f"📈 Loaded maximum strength scenario ({self.y.max():.2f} MPa)")
    
    def update_entry_from_slider(self, feature, value):
        if feature in self.entries:
            self.entries[feature].delete(0, tk.END)
            self.entries[feature].insert(0, f"{value:.3f}")
    
    def threaded_predict(self):
        self.predict_btn.config(state='disabled', text="⏳ Predicting...")
        self.status_left.config(text="⏳ Running prediction...")
        thread = threading.Thread(target=self.predict_strength)
        thread.daemon = True
        thread.start()
    
    def predict_strength(self):
        try:
            inputs = {}
            for feature, entry in self.entries.items():
                value = entry.get().strip()
                if value:
                    try:
                        inputs[feature] = float(value)
                    except:
                        inputs[feature] = self.feature_stats[feature]['mean']
                else:
                    inputs[feature] = self.feature_stats[feature]['mean']
            
            for feature in self.feature_names:
                if feature not in inputs:
                    inputs[feature] = self.feature_stats[feature]['mean']
            
            input_data = pd.DataFrame([inputs], columns=self.feature_names)
            prediction = self.model.predict(input_data)[0]
            
            confidence = 85
            for feature, value in inputs.items():
                stats = self.feature_stats[feature]
                if value < stats['min'] or value > stats['max']:
                    confidence -= 10
                elif value < stats['q1'] or value > stats['q3']:
                    confidence -= 5
            
            confidence = max(60, min(99, confidence))
            
            feature_contributions = {}
            predictions_with_variation = []
            
            for feature in self.feature_importance.head(5)['feature']:
                if feature in inputs:
                    temp_inputs = inputs.copy()
                    stats = self.feature_stats[feature]
                    temp_inputs[feature] = stats['mean']
                    temp_data = pd.DataFrame([temp_inputs], columns=self.feature_names)
                    base_pred = self.model.predict(temp_data)[0]
                    
                    contribution = (prediction - base_pred) * \
                                 self.feature_importance[self.feature_importance['feature'] == feature]['importance'].values[0] / 100
                    feature_contributions[feature] = contribution
                    
                    predictions_with_variation.append({
                        'feature': feature,
                        'value': inputs[feature],
                        'mean': stats['mean'],
                        'prediction': prediction
                    })
            
            self.root.after(0, lambda: self.display_result(
                prediction, confidence, inputs, feature_contributions, predictions_with_variation))
            
        except Exception as e:
            self.root.after(0, lambda: messagebox.showerror(
                "Prediction Error", f"❌ Prediction failed: {str(e)}"))
        
        self.root.after(0, self.enable_predict_button)
    
    def enable_predict_button(self):
        self.predict_btn.config(state='normal', text="🔮 PREDICT COMPRESSIVE STRENGTH")
        self.status_left.config(text="✅ Ready for predictions")
    
    def display_result(self, prediction, confidence, inputs, feature_contributions, comparisons):
        result_text = f"{prediction:.2f}"
        self.result_var.set(result_text)
        
        self.confidence_var.set(f"{confidence:.0f}%")
        self.confidence_bar['value'] = confidence
        
        classification, color = self.classify_strength(prediction)
        self.classification_var.set(classification)
        self.classification_label.config(fg=color)
        
        self.update_contributions_text(feature_contributions, prediction)
        self.update_comparison_text(comparisons, prediction)
        
        self.save_to_history(prediction, classification, inputs, confidence)
        self.status_left.config(text=f"✅ Prediction: {prediction:.2f} MPa - {classification}")
    
    def classify_strength(self, strength_value):
        percentiles = [
            self.y.quantile(0.25),
            self.y.quantile(0.50),
            self.y.quantile(0.75),
            self.y.quantile(0.90)
        ]
        
        if strength_value >= percentiles[3]:
            return "🔥 VERY HIGH STRENGTH", self.colors['danger']
        elif strength_value >= percentiles[2]:
            return "🔴 HIGH STRENGTH", self.colors['warning']
        elif strength_value >= percentiles[1]:
            return "🟠 MEDIUM STRENGTH", self.colors['accent']
        elif strength_value >= percentiles[0]:
            return "🟡 LOW STRENGTH", self.colors['success']
        else:
            return "🟢 VERY LOW STRENGTH", self.colors['extratrees_green']
    
    def update_contributions_text(self, contributions, prediction):
        self.contributions_text.config(state='normal')
        self.contributions_text.delete(1.0, tk.END)
        
        if contributions:
            text = f"📊 FEATURE CONTRIBUTIONS FOR STRENGTH: {prediction:.2f} MPa\n"
            text += "═"*60 + "\n\n"
            
            sorted_contrib = sorted(contributions.items(), key=lambda x: abs(x[1]), reverse=True)
            
            for feature, contribution in sorted_contrib:
                direction = "↑ INCREASES" if contribution > 0 else "↓ DECREASES"
                effect = abs(contribution)
                
                # Get display name
                display_name = feature
                for pred, col in self.found_predictors.items():
                    if col == feature:
                        display_info = {
                            'TGWP_replacement_ratio': 'TGWP Replacement',
                            'Water_to_binder_ratio': 'W/B Ratio',
                            'NaOH_concentration': 'NaOH Conc.',
                            'Curing_age': 'Curing Age'
                        }
                        display_name = display_info.get(pred, feature)
                        break
                
                bar_length = int(min(abs(effect) * 50, 20))
                bar = '█' * bar_length + '░' * (20 - bar_length)
                
                text += f"• {display_name:25} {direction:12} |{bar}| {contribution:+.4f} MPa\n"
                
                if abs(contribution) > 2.0:
                    text += f"  ⚡ Very strong {'positive' if contribution > 0 else 'negative'} impact\n"
                elif abs(contribution) > 1.0:
                    text += f"  📈 Strong {'positive' if contribution > 0 else 'negative'} impact\n"
                elif abs(contribution) > 0.5:
                    text += f"  📊 Moderate {'positive' if contribution > 0 else 'negative'} impact\n"
                else:
                    text += f"  📉 Minor impact\n"
            
            self.contributions_text.insert(1.0, text)
        else:
            self.contributions_text.insert(1.0, "No contribution data available.")
        
        self.contributions_text.config(state='disabled')
    
    def update_comparison_text(self, comparisons, prediction):
        self.comparison_text.config(state='normal')
        self.comparison_text.delete(1.0, tk.END)
        
        text = f"📈 COMPARISON WITH TYPICAL VALUES\n"
        text += "═"*60 + "\n\n"
        
        for comp in comparisons:
            # Get display name
            display_name = comp['feature']
            for pred, col in self.found_predictors.items():
                if col == comp['feature']:
                    display_info = {
                        'TGWP_replacement_ratio': 'TGWP Replacement',
                        'Water_to_binder_ratio': 'W/B Ratio',
                        'NaOH_concentration': 'NaOH Conc.',
                        'Curing_age': 'Curing Age'
                    }
                    display_name = display_info.get(pred, comp['feature'])
                    break
            
            text += f"• {display_name:25}\n"
            text += f"  Current: {comp['value']:.2f} | Typical: {comp['mean']:.2f}\n"
            diff = comp['value'] - comp['mean']
            if abs(diff) > 0.001:
                pct_diff = (diff / comp['mean']) * 100 if comp['mean'] != 0 else 0
                if diff > 0:
                    text += f"  📈 +{pct_diff:.1f}% above typical\n"
                else:
                    text += f"  📉 {pct_diff:.1f}% below typical\n"
            text += "\n"
        
        text += "\n🔍 INTERPRETATION:\n"
        text += "═"*40 + "\n"
        
        if prediction > self.y.quantile(0.75):
            text += "• High compressive strength - Suitable for structural applications\n"
            text += "• Mix design indicates good quality concrete\n"
        elif prediction > self.y.quantile(0.5):
            text += "• Moderate compressive strength - Suitable for general construction\n"
            text += "• Standard concrete mix design\n"
        elif prediction > self.y.quantile(0.25):
            text += "• Low compressive strength - Limited structural applications\n"
            text += "• May need mix optimization for better strength\n"
        else:
            text += "• Very low compressive strength - Non-structural applications only\n"
            text += "• Consider adjusting mix proportions for higher strength\n"
        
        self.comparison_text.insert(1.0, text)
        self.comparison_text.config(state='disabled')
    
    def save_to_history(self, prediction, classification, inputs, confidence):
        top_feature = self.feature_importance.iloc[0]['feature']
        top_value = inputs.get(top_feature, 0)
        
        history_entry = {
            'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'compressive_strength': float(prediction),
            'classification': classification,
            'confidence': float(confidence),
            'model': 'ExtraTrees_Tuned',
            'top_feature': f"{top_feature}: {top_value:.2f}",
            **{k: inputs[k] for k in self.feature_importance.head(3)['feature'].tolist() if k in inputs}
        }
        
        self.prediction_history.append(history_entry)
        self.update_history_display()
        self.save_history()
    
    def update_history_display(self):
        for item in self.history_tree.get_children():
            self.history_tree.delete(item)
        
        for entry in reversed(self.prediction_history[-100:]):
            self.history_tree.insert('', 'end', values=(
                entry['timestamp'],
                f"{entry['compressive_strength']:.2f}",
                entry['classification'],
                f"{entry['confidence']:.0f}%",
                entry.get('top_feature', 'N/A')
            ))
        
        if self.prediction_history:
            times = [h['compressive_strength'] for h in self.prediction_history]
            stats_text = (f"📊 Total Predictions: {len(self.prediction_history)} | "
                         f"Average: {np.mean(times):.2f} MPa | "
                         f"Range: {min(times):.2f} - {max(times):.2f} MPa")
            self.history_stats_var.set(stats_text)
    
    def load_history_entry(self, event):
        selection = self.history_tree.selection()
        if selection:
            item = self.history_tree.item(selection[0])
            values = item['values']
            timestamp = values[0]
            for entry in self.prediction_history:
                if entry['timestamp'] == timestamp:
                    for feature in self.entries:
                        if feature in entry:
                            self.entries[feature].delete(0, tk.END)
                            self.entries[feature].insert(0, f"{entry[feature]:.2f}")
                            for widget in self.entries[feature].master.winfo_children():
                                if isinstance(widget, ttk.Scale):
                                    widget.set(entry[feature])
                    
                    self.result_var.set(f"{entry['compressive_strength']:.2f}")
                    self.confidence_var.set(f"{entry['confidence']:.0f}%")
                    self.confidence_bar['value'] = entry['confidence']
                    self.classification_var.set(entry['classification'])
                    
                    self.status_left.config(text=f"✅ Loaded prediction from {timestamp}")
                    break
    
    def clear_inputs(self):
        for feature, entry in self.entries.items():
            entry.delete(0, tk.END)
            entry.insert(0, f"{self.feature_stats[feature]['mean']:.2f}")
            for widget in entry.master.winfo_children():
                if isinstance(widget, ttk.Scale):
                    widget.set(self.feature_stats[feature]['mean'])
        
        self.result_var.set("---")
        self.confidence_var.set("--%")
        self.confidence_bar['value'] = 0
        self.classification_var.set("Not predicted")
        
        self.contributions_text.config(state='normal')
        self.contributions_text.delete(1.0, tk.END)
        self.contributions_text.insert(1.0, "Feature contributions will appear here after prediction.")
        self.contributions_text.config(state='disabled')
        
        self.comparison_text.config(state='normal')
        self.comparison_text.delete(1.0, tk.END)
        self.comparison_text.insert(1.0, "Comparison with typical values will appear here.")
        self.comparison_text.config(state='disabled')
        
        self.status_left.config(text="🔄 Inputs cleared")
    
    def save_result(self):
        result_text = f"""═══════════════════════════════════════════════════════════════
              COMPRESSIVE STRENGTH PREDICTION
═══════════════════════════════════════════════════════════════

Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Model: Extra Trees Regressor (Tuned)
Predictors: {', '.join(self.feature_names)}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                    PREDICTION RESULT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Predicted Compressive Strength: {self.result_var.get()} MPa
Classification: {self.classification_var.get()}
Confidence: {self.confidence_var.get()}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                    INPUT PARAMETERS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

"""
        
        for feature in self.entries:
            # Get display name
            display_name = feature
            for pred, col in self.found_predictors.items():
                if col == feature:
                    display_info = {
                        'TGWP_replacement_ratio': 'TGWP Replacement',
                        'Water_to_binder_ratio': 'Water-to-Binder Ratio',
                        'NaOH_concentration': 'NaOH Concentration',
                        'Curing_age': 'Curing Age'
                    }
                    display_name = display_info.get(pred, feature)
                    break
            result_text += f"{display_name:30}: {self.entries[feature].get()}\n"
        
        result_text += f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                    MODEL INFORMATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Algorithm: Extra Trees Regressor
"""
        
        filename = filedialog.asksaveasfilename(
            defaultextension=".txt",
            filetypes=[("Text files", "*.txt"), ("All files", "*.*")],
            initialfile=f"strength_prediction_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        )
        
        if filename:
            try:
                with open(filename, 'w') as f:
                    f.write(result_text)
                messagebox.showinfo("Save Successful", f"✅ Result saved to {filename}")
                self.status_left.config(text=f"💾 Result saved to {os.path.basename(filename)}")
            except Exception as e:
                messagebox.showerror("Save Failed", f"❌ Failed to save: {str(e)}")
    
    def copy_to_clipboard(self):
        result_text = f"Compressive Strength: {self.result_var.get()} MPa ({self.classification_var.get()}) - Extra Trees Tuned Model"
        self.root.clipboard_clear()
        self.root.clipboard_append(result_text)
        messagebox.showinfo("Copied", "✅ Result copied to clipboard!")
        self.status_left.config(text="📋 Result copied to clipboard")
    
    def export_results(self):
        self.export_history()
    
    def clear_history(self):
        if not self.prediction_history:
            messagebox.showinfo("No History", "📭 Prediction history is already empty.")
            return
        
        if messagebox.askyesno("Clear History", "🗑️ Clear all prediction history?"):
            self.prediction_history = []
            self.update_history_display()
            self.save_history()
            messagebox.showinfo("History Cleared", "✅ Prediction history cleared.")
            self.status_left.config(text="🗑️ History cleared")
    
    def export_history(self):
        if not self.prediction_history:
            messagebox.showwarning("No Data", "⚠️ No prediction history to export.")
            return
        
        filename = filedialog.asksaveasfilename(
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv"), ("Excel files", "*.xlsx"), ("All files", "*.*")],
            initialfile=f"strength_predictions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        )
        
        if filename:
            try:
                df = pd.DataFrame(self.prediction_history)
                if filename.endswith('.xlsx'):
                    df.to_excel(filename, index=False)
                else:
                    df.to_csv(filename, index=False)
                messagebox.showinfo("Export Successful", f"✅ History exported to {filename}")
                self.status_left.config(text=f"📤 History exported to {os.path.basename(filename)}")
            except Exception as e:
                messagebox.showerror("Export Failed", f"❌ Failed to export: {str(e)}")
    
    def load_history_from_file(self):
        filename = filedialog.askopenfilename(
            filetypes=[("JSON files", "*.json"), ("CSV files", "*.csv"), ("All files", "*.*")]
        )
        
        if filename:
            try:
                if filename.endswith('.json'):
                    with open(filename, 'r') as f:
                        self.prediction_history = json.load(f)
                elif filename.endswith('.csv'):
                    self.prediction_history = pd.read_csv(filename).to_dict('records')
                
                self.update_history_display()
                messagebox.showinfo("Load Successful", 
                                   f"✅ Loaded {len(self.prediction_history)} predictions")
                self.status_left.config(text=f"📤 Loaded {len(self.prediction_history)} predictions from file")
            except Exception as e:
                messagebox.showerror("Load Failed", f"❌ Failed to load: {str(e)}")
    
    def save_history(self):
        try:
            with open(self.history_file, 'w') as f:
                json.dump(self.prediction_history, f, indent=2)
        except Exception as e:
            print(f"Failed to save history: {e}")
    
    def load_history(self):
        try:
            if os.path.exists(self.history_file):
                with open(self.history_file, 'r') as f:
                    self.prediction_history = json.load(f)
                self.update_history_display()
        except:
            pass
    
    # ========== PLOTTING FUNCTIONS ==========
    
    def save_plot(self, plot_function, filename_suffix):
        fig = plot_function(save_mode=True)
        if fig:
            filename = os.path.join(self.save_dir, f"{filename_suffix}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png")
            fig.savefig(filename, dpi=300, bbox_inches='tight')
            print(f"✅ Saved: {filename}")
            plt.close(fig)
            return True
        return False
    
    def save_all_plots(self):
        try:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            saved_plots = []
            
            batch_folder = os.path.join(self.save_dir, f"plots_{timestamp}")
            os.makedirs(batch_folder, exist_ok=True)
            original_save_dir = self.save_dir
            self.save_dir = batch_folder
            
            plot_functions = [
                (self.plot_feature_importance, "feature_importance"),
                (self.plot_data_distribution, "data_distribution"),
                (self.plot_partial_dependence, "partial_dependence"),
                (self.plot_correlation_matrix, "correlation_matrix"),
                (self.plot_feature_statistics, "feature_statistics"),
                (self.plot_actual_vs_predicted, "actual_vs_predicted"),
                (self.plot_mix_parameters, "mix_parameters"),
                (self.plot_residual_analysis, "residual_analysis"),
            ]
            
            for func, name in plot_functions:
                try:
                    if self.save_plot(func, name):
                        saved_plots.append(name)
                except Exception as e:
                    print(f"Failed to save {name}: {e}")
            
            self.save_dir = original_save_dir
            
            if saved_plots:
                messagebox.showinfo("Plots Saved", 
                                   f"✅ Successfully saved {len(saved_plots)} plots to:\n{batch_folder}")
                self.status_left.config(text=f"💾 Saved {len(saved_plots)} plots to folder")
            else:
                messagebox.showwarning("No Plots", "⚠️ No plots were generated to save.")
                
        except Exception as e:
            messagebox.showerror("Save Error", f"❌ Failed to save plots: {str(e)}")
    
    def plot_feature_importance(self, save_mode=False):
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(10, 6), dpi=100, facecolor='white')
        ax = fig.add_subplot(111)
        
        importance_df = self.feature_importance
        y_pos = np.arange(len(importance_df))
        colors = cm.YlOrBr(np.linspace(0.3, 0.9, len(importance_df)))
        
        bars = ax.barh(y_pos, importance_df['importance'], 
                      color=colors, edgecolor=self.colors['soil_brown'], 
                      height=0.5, linewidth=1.5)
        
        ax.set_yticks(y_pos)
        
        y_labels = []
        for feature in importance_df['feature']:
            for pred, col in self.found_predictors.items():
                if col == feature:
                    display_info = {
                        'TGWP_replacement_ratio': 'TGWP Replacement',
                        'Water_to_binder_ratio': 'W/B Ratio',
                        'NaOH_concentration': 'NaOH Conc.',
                        'Curing_age': 'Curing Age'
                    }
                    y_labels.append(display_info.get(pred, feature[:20]))
                    break
            else:
                y_labels.append(feature[:20])
        
        ax.set_yticklabels(y_labels, fontsize=11)
        ax.set_xlabel('Feature Importance Score', fontsize=12, fontweight='bold')
        ax.set_title(f'Feature Importance for {self.target_name}\nExtra Trees (Tuned)', 
                    fontsize=14, fontweight='bold', pad=20, color=self.colors['soil_brown'])
        ax.grid(True, alpha=0.3, axis='x', linestyle='--')
        
        total_importance = importance_df['importance'].sum()
        for i, (bar, importance) in enumerate(zip(bars, importance_df['importance'])):
            percentage = (importance / total_importance) * 100
            ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                   f'{percentage:.1f}%', va='center', fontsize=10, fontweight='bold')
        
        fig.tight_layout()
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def plot_data_distribution(self, save_mode=False):
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(10, 6), dpi=100, facecolor='white')
        ax = fig.add_subplot(111)
        
        n, bins, patches = ax.hist(self.y, bins=20, alpha=0.7, 
                                  color=self.colors['sandy'], 
                                  edgecolor=self.colors['soil_brown'], 
                                  density=True, label='Histogram')
        
        from scipy import stats
        kde = stats.gaussian_kde(self.y)
        x_range = np.linspace(self.y.min(), self.y.max(), 200)
        ax.plot(x_range, kde(x_range), color=self.colors['danger'], 
               linewidth=2.5, label='Density Curve')
        
        ax.set_xlabel(f'{self.target_name} (MPa)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Density', fontsize=12, fontweight='bold')
        ax.set_title(f'Distribution of {self.target_name}', 
                    fontsize=14, fontweight='bold', pad=20, color=self.colors['soil_brown'])
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend()
        
        stats_text = (f'Mean: {self.y.mean():.2f} MPa\n'
                     f'Std: {self.y.std():.2f} MPa\n'
                     f'Min: {self.y.min():.2f} MPa\n'
                     f'25%: {self.y.quantile(0.25):.2f} MPa\n'
                     f'50%: {self.y.median():.2f} MPa\n'
                     f'75%: {self.y.quantile(0.75):.2f} MPa\n'
                     f'Max: {self.y.max():.2f} MPa\n'
                     f'Samples: {len(self.y)}')
        
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
               fontsize=9, verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        fig.tight_layout()
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def plot_partial_dependence(self, save_mode=False):
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(12, 10), dpi=100, facecolor='white')
        
        for i, feature in enumerate(self.feature_names, 1):
            ax = fig.add_subplot(2, 2, i)
            
            try:
                pdp = partial_dependence(self.model, self.X, [feature], 
                                        grid_resolution=50)
                
                ax.plot(pdp['values'][0], pdp['average'][0], 
                       linewidth=2.5, color=self.colors['soil_brown'], alpha=0.8)
                ax.fill_between(pdp['values'][0], 
                               pdp['average'][0] - self.rmse/2,
                               pdp['average'][0] + self.rmse/2,
                               alpha=0.2, color=self.colors['sandy'])
                
                display_name = feature
                for pred, col in self.found_predictors.items():
                    if col == feature:
                        display_info = {
                            'TGWP_replacement_ratio': 'TGWP Replacement (%)',
                            'Water_to_binder_ratio': 'Water-to-Binder Ratio',
                            'NaOH_concentration': 'NaOH Concentration (M)',
                            'Curing_age': 'Curing Age (days)'
                        }
                        display_name = display_info.get(pred, feature)
                        break
                
                ax.set_xlabel(display_name, fontsize=10, fontweight='bold')
                ax.set_ylabel('Partial Dependence (MPa)', fontsize=10, fontweight='bold')
                ax.set_title(f'PDP: {display_name}', fontsize=11, fontweight='bold')
                ax.grid(True, alpha=0.3, linestyle='--')
                
            except Exception as e:
                ax.text(0.5, 0.5, f"Error plotting", 
                       ha='center', va='center', fontsize=10, color='red')
        
        fig.suptitle(f'Partial Dependence Plots for All Predictors\nExtra Trees (Tuned)', 
                    fontsize=14, fontweight='bold', y=0.98, color=self.colors['soil_brown'])
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def plot_correlation_matrix(self, save_mode=False):
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(8, 6), dpi=100, facecolor='white')
        ax = fig.add_subplot(111)
        
        corr_features = self.feature_names + [self.target_name]
        corr_matrix = self.df[corr_features].corr()
        
        im = ax.imshow(corr_matrix, cmap='YlOrBr', vmin=-1, vmax=1, aspect='auto')
        
        display_names = []
        for col in corr_features:
            found = False
            for pred, col_name in self.found_predictors.items():
                if col_name == col:
                    display_info = {
                        'TGWP_replacement_ratio': 'TGWP Replacement',
                        'Water_to_binder_ratio': 'W/B Ratio',
                        'NaOH_concentration': 'NaOH Conc.',
                        'Curing_age': 'Curing Age'
                    }
                    display_names.append(display_info.get(pred, col[:10]))
                    found = True
                    break
            if not found:
                display_names.append('Strength')
        
        ax.set_xticks(np.arange(len(corr_features)))
        ax.set_yticks(np.arange(len(corr_features)))
        ax.set_xticklabels(display_names, rotation=45, ha='right', fontsize=9)
        ax.set_yticklabels(display_names, fontsize=9)
        
        ax.set_title(f'Correlation Matrix\nValidated Predictors', 
                    fontsize=14, fontweight='bold', pad=20, color=self.colors['soil_brown'])
        
        for i in range(len(corr_features)):
            for j in range(len(corr_features)):
                text = ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                             ha="center", va="center", color="black", fontsize=8)
        
        cbar = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.set_ylabel('Correlation Coefficient', rotation=-90, va="bottom", fontsize=10)
        
        fig.tight_layout()
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def plot_feature_statistics(self, save_mode=False):
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(12, 8), dpi=100, facecolor='white')
        
        for i, feature in enumerate(self.feature_names, 1):
            ax = fig.add_subplot(2, 2, i)
            data = self.X[feature].dropna()
            
            bp = ax.boxplot(data, vert=True, patch_artist=True, 
                           widths=0.6, showfliers=True)
            
            bp['boxes'][0].set_facecolor(self.colors['sandy'])
            bp['boxes'][0].set_alpha(0.7)
            bp['boxes'][0].set_edgecolor(self.colors['soil_brown'])
            bp['boxes'][0].set_linewidth(2)
            bp['whiskers'][0].set_color(self.colors['soil_brown'])
            bp['whiskers'][1].set_color(self.colors['soil_brown'])
            bp['caps'][0].set_color(self.colors['soil_brown'])
            bp['caps'][1].set_color(self.colors['soil_brown'])
            bp['medians'][0].set_color(self.colors['danger'])
            
            display_name = feature
            for pred, col in self.found_predictors.items():
                if col == feature:
                    display_info = {
                        'TGWP_replacement_ratio': 'TGWP Replacement (%)',
                        'Water_to_binder_ratio': 'W/B Ratio',
                        'NaOH_concentration': 'NaOH Conc. (M)',
                        'Curing_age': 'Curing Age (days)'
                    }
                    display_name = display_info.get(pred, feature)
                    break
            
            ax.set_ylabel(display_name, fontsize=10, fontweight='bold')
            ax.set_title(f'Distribution of {display_name}', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y', linestyle='--')
            
            stats_text = f'n={len(data)}\nμ={data.mean():.2f}\nσ={data.std():.2f}'
            ax.text(1.1, 0.5, stats_text, transform=ax.transAxes,
                   fontsize=8, verticalalignment='center',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        fig.suptitle(f'Statistical Distribution of Validated Predictors', 
                    fontsize=14, fontweight='bold', y=0.98, color=self.colors['soil_brown'])
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def plot_actual_vs_predicted(self, save_mode=False):
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(10, 8), dpi=100, facecolor='white')
        ax = fig.add_subplot(111)
        
        predictions = self.model.predict(self.X)
        
        scatter = ax.scatter(self.y, predictions, 
                            alpha=0.6, 
                            s=80, 
                            c=self.colors['sandy'],
                            edgecolors=self.colors['soil_brown'], 
                            linewidth=0.5,
                            label='Predictions')
        
        min_val = min(self.y.min(), predictions.min())
        max_val = max(self.y.max(), predictions.max())
        ax.plot([min_val, max_val], [min_val, max_val], 
               'r--', linewidth=2, alpha=0.7, label='Perfect Prediction')
        
        ax.fill_between([min_val, max_val], 
                       [min_val - self.rmse, max_val - self.rmse],
                       [min_val + self.rmse, max_val + self.rmse],
                       alpha=0.1, color='gray', label=f'±RMSE ({self.rmse:.2f} MPa)')
        
        ax.set_xlabel(f'Actual {self.target_name} (MPa)', fontsize=12, fontweight='bold')
        ax.set_ylabel(f'Predicted {self.target_name} (MPa)', fontsize=12, fontweight='bold')
        ax.set_title(f'Actual vs Predicted {self.target_name}\nExtra Trees (Tuned)', 
                    fontsize=14, fontweight='bold', pad=20, color=self.colors['soil_brown'])
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(fontsize=10)
        
        fig.tight_layout()
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def plot_mix_parameters(self, save_mode=False):
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(12, 10), dpi=100, facecolor='white')
        
        for i, feature in enumerate(self.feature_names, 1):
            ax = fig.add_subplot(2, 2, i)
            
            ax.scatter(self.X[feature], self.y, 
                      alpha=0.5, s=40, c=self.colors['sandy'], 
                      edgecolors=self.colors['soil_brown'], linewidth=0.5)
            
            z = np.polyfit(self.X[feature], self.y, 1)
            p = np.poly1d(z)
            x_range = np.linspace(self.X[feature].min(), self.X[feature].max(), 100)
            ax.plot(x_range, p(x_range), "r--", alpha=0.8, linewidth=2)
            
            display_name = feature
            for pred, col in self.found_predictors.items():
                if col == feature:
                    display_info = {
                        'TGWP_replacement_ratio': 'TGWP Replacement (%)',
                        'Water_to_binder_ratio': 'W/B Ratio',
                        'NaOH_concentration': 'NaOH Conc. (M)',
                        'Curing_age': 'Curing Age (days)'
                    }
                    display_name = display_info.get(pred, feature)
                    break
            
            ax.set_xlabel(display_name, fontsize=10, fontweight='bold')
            ax.set_ylabel('Strength (MPa)', fontsize=10, fontweight='bold')
            ax.set_title(f'{display_name} vs Strength', fontsize=11, fontweight='bold')
            ax.grid(True, alpha=0.3, linestyle='--')
            
            corr = self.X[feature].corr(self.y)
            ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes,
                   fontsize=9, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        fig.suptitle('Validated Predictors vs Compressive Strength', 
                    fontsize=14, fontweight='bold', y=0.98, color=self.colors['soil_brown'])
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def plot_residual_analysis(self, save_mode=False):
        if not save_mode:
            self.clear_analysis_frame()
        
        fig = Figure(figsize=(12, 10), dpi=100, facecolor='white')
        
        predictions = self.model.predict(self.X)
        residuals = self.y - predictions
        
        ax1 = fig.add_subplot(2, 2, 1)
        ax1.scatter(predictions, residuals, alpha=0.5, s=40,
                   c=self.colors['sandy'], edgecolors=self.colors['soil_brown'], linewidth=0.5)
        ax1.axhline(y=0, color='r', linestyle='--', linewidth=2)
        ax1.set_xlabel('Predicted Values (MPa)', fontsize=10, fontweight='bold')
        ax1.set_ylabel('Residuals (MPa)', fontsize=10, fontweight='bold')
        ax1.set_title('Residuals vs Predicted', fontsize=11, fontweight='bold')
        ax1.grid(True, alpha=0.3, linestyle='--')
        
        ax2 = fig.add_subplot(2, 2, 2)
        ax2.hist(residuals, bins=15, alpha=0.7, 
                color=self.colors['sandy'], edgecolor=self.colors['soil_brown'])
        ax2.axvline(x=0, color='r', linestyle='--', linewidth=2)
        ax2.set_xlabel('Residuals (MPa)', fontsize=10, fontweight='bold')
        ax2.set_ylabel('Frequency', fontsize=10, fontweight='bold')
        ax2.set_title('Distribution of Residuals', fontsize=11, fontweight='bold')
        ax2.grid(True, alpha=0.3, linestyle='--')
        
        ax3 = fig.add_subplot(2, 2, 3)
        from scipy import stats
        stats.probplot(residuals, dist="norm", plot=ax3)
        ax3.set_title('Q-Q Plot', fontsize=11, fontweight='bold')
        ax3.grid(True, alpha=0.3, linestyle='--')
        
        ax4 = fig.add_subplot(2, 2, 4)
        top_feature = self.feature_importance.iloc[0]['feature']
        ax4.scatter(self.X[top_feature], residuals, alpha=0.5, s=40,
                   c=self.colors['sandy'], edgecolors=self.colors['soil_brown'], linewidth=0.5)
        ax4.axhline(y=0, color='r', linestyle='--', linewidth=2)
        
        display_name = top_feature
        for pred, col in self.found_predictors.items():
            if col == top_feature:
                display_info = {
                    'TGWP_replacement_ratio': 'TGWP Replacement',
                    'Water_to_binder_ratio': 'W/B Ratio',
                    'NaOH_concentration': 'NaOH Conc.',
                    'Curing_age': 'Curing Age'
                }
                display_name = display_info.get(pred, top_feature[:15])
                break
        
        ax4.set_xlabel(display_name, fontsize=10, fontweight='bold')
        ax4.set_ylabel('Residuals (MPa)', fontsize=10, fontweight='bold')
        ax4.set_title(f'Residuals vs {display_name}', fontsize=11, fontweight='bold')
        ax4.grid(True, alpha=0.3, linestyle='--')
        
        fig.suptitle('Residual Analysis - Extra Trees (Tuned)', 
                    fontsize=14, fontweight='bold', y=0.98, color=self.colors['soil_brown'])
        fig.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        if not save_mode:
            canvas = FigureCanvasTkAgg(fig, self.analysis_frame)
            canvas.draw()
            widget = canvas.get_tk_widget()
            widget.pack(fill='both', expand=True)
            self.current_plot = fig
        
        return fig
    
    def clear_analysis_frame(self):
        for widget in self.analysis_frame.winfo_children():
            widget.destroy()
    
    def run(self):
        print("\n" + "="*70)
        print("🏗️ COMPRESSIVE STRENGTH PREDICTOR - EXTRA TREES (TUNED)")
        print("="*70)
        print(f"📊 Target Variable: {self.target_name}")
        print(f"🔢 Number of Predictors: {len(self.feature_names)}")
        print(f"🌲 Algorithm: Extra Trees Regressor with tuned hyperparameters")
        print(f"💾 Plot save directory: {self.save_dir}")
        print("="*70)
        print("✅ Application ready! Starting GUI...")
        print("🏁 Press Ctrl+C in terminal to stop\n")
        
        self.root.after(100, lambda: self.load_preset_by_name("Medium Strength Mix"))
        self.root.mainloop()

# Run the application
if __name__ == "__main__":
    try:
        print("🏗️ Initializing Compressive Strength Predictor...")
        app = CompressiveStrengthPredictor()
        app.run()
    except KeyboardInterrupt:
        print("\n👋 Application terminated by user")
    except Exception as e:
        print(f"❌ Application Error: {str(e)}")
        import traceback
        traceback.print_exc()
        messagebox.showerror("Application Error", f"Failed to start:\n\n{str(e)}")

🏗️ Initializing Compressive Strength Predictor...
📂 Loading data from: D:\2026 Work\Dr Hariharan Surendran\Paper 1 CS Glass Waste\Revision Work\Data\Data.csv
✅ Dataset loaded successfully!
📐 Dataset shape: (192, 9)
🔧 Cleaned columns: ['Fly_ash_Kg_m3', 'Toughened_glass_waste_powder_Kg_m3', 'Replacement', 'M_sand', 'W_b_ratio_Kg_m3', 'water_Kg_m3', 'Na2SiO3_NaOH_Molarity_m', 'Age_Days', 'Compressive_strength_Mpa']
✅ Found 'TGWP_replacement_ratio' → 'Toughened_glass_waste_powder_Kg_m3' (by keyword)
✅ Found 'Water_to_binder_ratio' → 'W_b_ratio_Kg_m3' (by keyword)
✅ Found 'NaOH_concentration' → 'Na2SiO3_NaOH_Molarity_m' (by keyword)
✅ Found 'Curing_age' → 'Age_Days' (by keyword)
✅ Target column found: 'Compressive_strength_Mpa'
🎯 Target variable: 'Compressive_strength_Mpa'

📊 Using 4 predictors:
   • TGWP_replacement_ratio: 'Toughened_glass_waste_powder_Kg_m3'
   • Water_to_binder_ratio: 'W_b_ratio_Kg_m3'
   • NaOH_concentration: 'Na2SiO3_NaOH_Molarity_m'
   • Curing_age: 'Age_Days'

📊 Targ